In [1]:
TOTAL_KEYWORDS = [
    # English
    'total', 'grand total', 'total amount', 'amount due', 'amount payable',
    'net total', 'subtotal', 'balance due', 'total payable',
    # Payment method lines (often == total charged)
    'debit card', 'credit card', 'kredit card', 'cash', 'tunai',
    'e-wallet', 'touch n go', 'grab pay', 'boost', 'maybank qr',
    # Malay / Malaysian formal
    'jumlah', 'jumlah besar', 'jumlah bayaran', 'jumlah keseluruhan',
    'amaun', 'amaun bayaran', 'amaun perlu dibayar', 'amaun dibayar',
    'baki', 'baki perlu dibayar', 'baki tertunggak',
    'bayaran', 'dibayar', 'diterima',
    # Common abbreviations / shortforms on thermal prints
    'jml', 'amt', 'ttl',
]

In [2]:
import re

AMOUNT_PATTERN = r'\d{1,6}[.,]\d{2}'  # tightened upper bound, avoids matching phone numbers

def find_total(results_detail, keywords, y_tolerance=80):
    """
    results_detail : EasyOCR output with detail=1
    keywords       : TOTAL_KEYWORDS list
    y_tolerance    : max pixel distance to consider a keyword-amount pair valid
                     (guards against false matches on long receipts)
    """
    keyword_lines = []
    amount_lines = []

    for bbox, text, conf in results_detail:
        top_y = min(p[1] for p in bbox)
        text_lower = text.lower()

        if any(kw in text_lower for kw in keywords):
            keyword_lines.append((top_y, text))

        if re.search(AMOUNT_PATTERN, text):
            amount_lines.append((top_y, text, conf))

    # --- Edge case: nothing extracted at all ---
    if not amount_lines:
        return {
            'status': 'no_amounts_found',
            'total': None,
            'matched_keyword': None,
            'confidence': None,
            'raw_line': None,
            'flag_for_review': True,
        }

    # --- Edge case: no keyword found — fall back to last amount on receipt ---
    if not keyword_lines:
        fallback = max(amount_lines, key=lambda x: x[0])  # lowest on page = last line
        raw_amount = re.search(AMOUNT_PATTERN, fallback[1]).group()
        return {
            'status': 'keyword_not_found_used_last_amount',
            'total': raw_amount,
            'matched_keyword': None,
            'confidence': fallback[2],
            'raw_line': fallback[1],
            'flag_for_review': True,  # still flag — human should verify
        }

    # --- Main path: keyword found, find spatially closest amount ---
    results_out = []
    for kw_y, kw_text in keyword_lines:
        candidates = [(abs(a[0] - kw_y), a) for a in amount_lines]
        dist, closest = min(candidates, key=lambda x: x[0])

        if dist > y_tolerance:
            # keyword found but no amount nearby — suspicious, flag it
            results_out.append({
                'status': 'keyword_found_no_nearby_amount',
                'total': None,
                'matched_keyword': kw_text,
                'confidence': None,
                'raw_line': None,
                'flag_for_review': True,
            })
            continue

        raw_amount = re.search(AMOUNT_PATTERN, closest[1]).group()
        results_out.append({
            'status': 'ok',
            'total': raw_amount,
            'matched_keyword': kw_text,
            'confidence': closest[2],
            'raw_line': closest[1],
            'flag_for_review': closest[2] < 0.6,  # flag low-confidence reads
        })

    # If multiple keywords matched, prefer highest-confidence ok result
    ok_results = [r for r in results_out if r['status'] == 'ok']
    if ok_results:
        return max(ok_results, key=lambda x: x['confidence'])

    # All keyword matches were suspicious
    return results_out[0]

In [3]:
import easyocr
import cv2

reader = easyocr.Reader(['en'])  # initialise once, reuse

def extract_total(image_path: str, upscale: bool = True) -> dict:
    """
    Main entry point.
    Returns a dict with keys: status, total, matched_keyword,
                               confidence, raw_line, flag_for_review
    """
    img = cv2.imread(image_path)
    if img is None:
        return {
            'status': 'image_load_failed',
            'total': None,
            'matched_keyword': None,
            'confidence': None,
            'raw_line': None,
            'flag_for_review': True,
        }

    if upscale:
        img = cv2.resize(img, None, fx=2, fy=2, interpolation=cv2.INTER_LINEAR)

    results_detail = reader.readtext(img, detail=1)
    return find_total(results_detail, TOTAL_KEYWORDS)


# --- Usage ---
result = extract_total("D:\LENOVO\Downloads\WhatsApp Image 2026-06-20 at 00.59.23.jpeg")
print(result)

# Quick display helper
if result['flag_for_review']:
    print(f"⚠️  FLAG FOR REVIEW — status: {result['status']}")
else:
    print(f"✅  Total: {result['total']}  (keyword: '{result['matched_keyword']}', conf: {result['confidence']:.2f})")

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.
c:\Users\LENOVO\Desktop\OCR project\.venv\lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'status': 'keyword_found_no_nearby_amount', 'total': None, 'matched_keyword': 'TOTAL', 'confidence': None, 'raw_line': None, 'flag_for_review': True}
⚠️  FLAG FOR REVIEW — status: keyword_found_no_nearby_amount
